# 數據準備與特徵工程

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明數據清理在機器學習流程中的重要性。
2. 使用 pandas 與 scikit-learn 處理缺失值、異常值與重複資料。
3. 比較不同資料尺度調整方法的適用情境。
4. 對類別欄位進行 One-hot Encoding 與序位編碼。
5. 使用特徵選擇與 PCA 降維改善模型輸入品質。

本練習以一個簡化的客戶流失預測資料集為例，示範從原始資料到可建模特徵矩陣的完整準備流程。


In [ ]:
# ── 環境設定與範例資料建立 ─────────────────────────────
# 載入本章節所需套件，並建立一份包含缺失值、異常值、重複資料、數值欄位與類別欄位的範例資料。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

n = 120
df = pd.DataFrame({
    "age": np.random.normal(38, 10, n).round(),
    "monthly_fee": np.random.normal(1200, 300, n).round(1),
    "usage_hours": np.random.exponential(35, n).round(1),
    "plan": np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.45, 0.35, 0.20]),
    "region": np.random.choice(["北部", "中部", "南部"], n),
    "satisfaction": np.random.choice(["低", "中", "高"], n, p=[0.25, 0.45, 0.30])
})

# 建立目標欄位：是否流失
score = (
    -0.04 * df["age"]
    + 0.003 * df["monthly_fee"]
    - 0.02 * df["usage_hours"]
    + df["satisfaction"].map({"低": 1.2, "中": 0.4, "高": -0.6})
    + np.random.normal(0, 0.6, n)
)
prob = 1 / (1 + np.exp(-score))
df["churn"] = (prob > np.median(prob)).astype(int)

# 製造缺失值、異常值與重複列
for col in ["age", "monthly_fee", "usage_hours", "plan"]:
    missing_idx = np.random.choice(df.index, size=5, replace=False)
    df.loc[missing_idx, col] = np.nan

df.loc[3, "monthly_fee"] = 9800
df.loc[8, "usage_hours"] = 450
df = pd.concat([df, df.iloc[[0, 1]]], ignore_index=True)

print("資料筆數與欄位數：", df.shape)
print("缺失值數量：")
print(df.isna().sum())
print("\n前 5 筆資料：")
print(df.head())


## 核心概念說明

在機器學習建模前，原始資料通常不能直接送入模型。數據準備與特徵工程的主要任務，是將資料轉換成一致、可解釋、可計算且適合模型學習的形式。

### 1. 數據清理

常見清理工作包括：

- 缺失值處理：可使用刪除法、平均數、中位數、眾數、KNN 或模型預測填補。
- 異常值處理：可使用 Z-score、IQR、箱型圖、Isolation Forest 等方法偵測。
- 重複樣本處理：避免模型被重複觀測值過度影響。
- 格式與單位一致化：例如日期格式、類別名稱、公斤與公克等單位統一。

### 2. 特徵轉換

不同模型對資料尺度與型態的敏感度不同。線性模型、SVM、KNN、神經網路通常需要標準化；樹模型對尺度較不敏感，但仍需要合理處理缺失值與類別欄位。

### 3. 特徵選擇與降維

當特徵過多時，可能造成訓練時間增加、模型過度擬合或解釋困難。可使用 Filter、Wrapper、Embedded 方法進行特徵選擇，也可使用 PCA 等方法進行降維。


In [ ]:
# ── 示範：缺失值、重複資料與異常值處理 ───────────────────────
# 這段程式碼示範如何移除重複資料、用中位數與眾數填補缺失值，並用 IQR 方法標記與截尾異常值。

import numpy as np
import pandas as pd

np.random.seed(42)
n = 120
df = pd.DataFrame({
    "age": np.random.normal(38, 10, n).round(),
    "monthly_fee": np.random.normal(1200, 300, n).round(1),
    "usage_hours": np.random.exponential(35, n).round(1),
    "plan": np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.45, 0.35, 0.20]),
    "region": np.random.choice(["北部", "中部", "南部"], n),
    "satisfaction": np.random.choice(["低", "中", "高"], n, p=[0.25, 0.45, 0.30])
})
df["churn"] = np.random.choice([0, 1], n)
for col in ["age", "monthly_fee", "usage_hours", "plan"]:
    df.loc[np.random.choice(df.index, size=5, replace=False), col] = np.nan
df.loc[3, "monthly_fee"] = 9800
df.loc[8, "usage_hours"] = 450
df = pd.concat([df, df.iloc[[0, 1]]], ignore_index=True)

print("清理前資料筆數：", len(df))
df_clean = df.drop_duplicates().copy()
print("移除重複後資料筆數：", len(df_clean))

numeric_cols = ["age", "monthly_fee", "usage_hours"]
categorical_cols = ["plan", "region", "satisfaction"]

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# 使用 IQR 偵測並截尾異常值
for col in ["monthly_fee", "usage_hours"]:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_clean[col + "_is_outlier"] = ((df_clean[col] < lower) | (df_clean[col] > upper)).astype(int)
    df_clean[col] = df_clean[col].clip(lower, upper)

print("\n清理後缺失值數量：")
print(df_clean.isna().sum())
print("\n異常值標記數量：")
print(df_clean[["monthly_fee_is_outlier", "usage_hours_is_outlier"]].sum())
print("\n清理後摘要統計：")
print(df_clean[numeric_cols].describe().round(2))


## 特徵轉換與標準化

特徵轉換的目標，是讓模型接收到更合理的數值表達。

### 數值尺度調整

- Min-Max Normalization：將數值縮放到 0 到 1，容易解釋，但對極端值敏感。
- Z-score Standardization：轉換為平均值 0、標準差 1，常用於線性模型、SVM、KNN。
- Robust Scaling：使用中位數與 IQR，適合含有異常值或偏態分佈的資料。

### 類別資料編碼

- Label Encoding 或 Ordinal Encoding：適合有順序的類別，例如滿意度「低、中、高」。
- One-hot Encoding：適合無順序的類別，例如地區、產品類型、方案名稱。

### 實務提醒

資料轉換器應只在訓練資料上 fit，再用同一組轉換規則 transform 測試資料。這可以避免資料洩漏，確保模型評估較接近真實部署情境。


In [ ]:
# ── 示範：類別編碼與資料標準化 Pipeline ──────────────────
# 這段程式碼使用 ColumnTransformer 同時處理數值欄位與類別欄位，建立可重現的前處理流程。

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 160
df = pd.DataFrame({
    "age": np.random.normal(38, 10, n).round(),
    "monthly_fee": np.random.normal(1200, 300, n).round(1),
    "usage_hours": np.random.exponential(35, n).round(1),
    "plan": np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.45, 0.35, 0.20]),
    "region": np.random.choice(["北部", "中部", "南部"], n),
    "satisfaction": np.random.choice(["低", "中", "高"], n, p=[0.25, 0.45, 0.30])
})

score = (
    -0.04 * df["age"]
    + 0.003 * df["monthly_fee"]
    - 0.02 * df["usage_hours"]
    + df["satisfaction"].map({"低": 1.2, "中": 0.4, "高": -0.6})
    + np.random.normal(0, 0.6, n)
)
prob = 1 / (1 + np.exp(-score))
df["churn"] = (prob > np.median(prob)).astype(int)

for col in ["age", "monthly_fee", "usage_hours", "plan"]:
    df.loc[np.random.choice(df.index, size=6, replace=False), col] = np.nan

X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_features = ["age", "monthly_fee", "usage_hours"]
ordinal_features = ["satisfaction"]
nominal_features = ["plan", "region"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["低", "中", "高"]]))
])

nominal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("ord", ordinal_pipeline, ordinal_features),
    ("nom", nominal_pipeline, nominal_features)
])

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("訓練資料筆數：", X_train.shape[0])
print("測試資料筆數：", X_test.shape[0])
print("測試集準確率：", round(accuracy_score(y_test, y_pred), 3))
print("轉換後特徵數：", model.named_steps["preprocess"].transform(X_train).shape[1])
